# 02 · Data analysis
Descriptive statistics of the processed CMSB, EDOS and EXIST 2025 files written by `01_data_prep`.
- **Inputs:** `data/processed/{train,dev,test}/{cmsb,edos,exist}_{split}.parquet`, plus the task spec in `configs/data_prep.yaml`.
- **Outputs:** CSV tables, PNG figures, `metrics.json`, the config and a README in `OUTPUT_ROOT/{run_name}/`.

**Test files are opened label-blind.** Only the columns in `TEST_COLS` are read from them (ids, normalised text, language and audit flags), so test labels never reach memory. CMSB `annotated` and `n_annot` stay out too: they mark the all-non-sexist `other` source, which gives the label away. Every label statistic below is for train and dev only.

No post text is printed.

In [ ]:
import os
import shutil
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Sexism-research")  # TODO: set to your Drive clone
else:
    PROJECT_ROOT = Path.cwd().parent  # local run from notebooks/
DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
HF_HOME = PROJECT_ROOT / "hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)

In [ ]:
# 00's pins plus matplotlib for the figures
PINS = {"pandas": "3.0.6", "numpy": "2.5.4", "pyarrow": "26.0.0", "pyyaml": "6.0.3", "matplotlib": "3.11.2"}
if IN_COLAB:
    PIP_ARGS = " ".join(f"{p}=={v}" for p, v in PINS.items())
    %pip install -q {PIP_ARGS}
    print("If pip changed anything, restart the runtime, then run all again")

In [ ]:
import datetime
import hashlib
import json

import yaml

CONFIG = {
    "dataset": "all",
    "task": "analysis",
    "model": "none",
    "method": "eda",
    "seed": 42,
    "datasets": ["cmsb", "edos", "exist"],
    "splits": ["train", "dev", "test"],
    "length_clip": 70,  # words; longer posts go in the last histogram bin
}
CONFIG["tasks"] = yaml.safe_load((PROJECT_ROOT / "configs" / "data_prep.yaml").read_text())["tasks"]
CONFIG["run_name"] = (
    f"{datetime.datetime.now():%Y%m%d-%H%M}_{CONFIG['dataset']}_{CONFIG['task']}"
    f"_{CONFIG['model']}_{CONFIG['method']}_s{CONFIG['seed']}"
)
RUN_DIR = OUTPUT_ROOT / CONFIG["run_name"]
# A finished run is never overwritten
assert not RUN_DIR.exists(), f"{CONFIG['run_name']} exists: wait a minute and rerun"
(RUN_DIR / "figures").mkdir(parents=True)
print(CONFIG["run_name"])

In [ ]:
import random

import numpy as np

# Nothing below is random; seeded so every notebook has the same cell order
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

In [ ]:
import platform
import re
from importlib.metadata import distributions, version

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq

ENV = {"python": platform.python_version(), **{p: version(p) for p in PINS}}
print(ENV)
off_pin = {p: ENV[p] for p in PINS if ENV[p] != PINS[p]}
assert not off_pin, f"installed {off_pin} differ from PINS: install the pins and restart the kernel"
PKG_FREEZE = "\n".join(sorted({f"{d.metadata['Name']}=={d.version}" for d in distributions()}, key=str.lower)) + "\n"
# Hash of the notebook as saved on disk: save before running so it matches what ran
NOTEBOOK_SHA = hashlib.sha256((PROJECT_ROOT / "notebooks" / "02_data_analysis.ipynb").read_bytes()).hexdigest()

# Reference palette slots 1-3 (they stay distinguishable for every pair, colour-blind included); one colour per split
SPLIT_COLOR = {"train": "#2a78d6", "dev": "#eb6834", "test": "#1baf7a"}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e4e3df", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.edgecolor": "#8a8984", "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e",
})

## Load
On Colab the files are copied to local disk first. Each file's `content_sha256` (set by 01) is recorded, so the stats can be traced to an exact build. Test files are read with `TEST_COLS` only.

In [ ]:
TEST_COLS = {
    "cmsb": ["id", "text_norm", "lang", "empty_norm", "dup_group", "leak_group", "of_id"],
    "edos": ["id", "text_norm", "lang", "empty_norm", "dup_group", "leak_group"],
    "exist": ["id", "text_norm", "lang", "empty_norm", "dup_group", "leak_group"],
}
LABEL_COLS = {"sexist", "sexist_soft", "n_annot", "annotated", "category", "vector", "labels"}
assert not any(LABEL_COLS & set(c) for c in TEST_COLS.values()), "a label column is in TEST_COLS"

SRC = DATA_ROOT / "processed"
LOCAL = Path("/content/data/processed") if IN_COLAB else SRC
DF, CONTENT_SHA = {}, {}
for ds in CONFIG["datasets"]:
    for s in CONFIG["splits"]:
        rel = Path(s) / f"{ds}_{s}.parquet"
        if IN_COLAB:
            (LOCAL / rel).parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(SRC / rel, LOCAL / rel)
        meta = pq.read_schema(LOCAL / rel).metadata or {}
        CONTENT_SHA[f"{ds}_{s}"] = json.loads(meta[b"heft"]).get("content_sha256") if b"heft" in meta else None
        DF[ds, s] = pd.read_parquet(LOCAL / rel, columns=TEST_COLS[ds] if s == "test" else None)
assert all(c in TEST_COLS[ds] for (ds, s), d in DF.items() if s == "test" for c in d.columns)
print({k: v[:12] if v else None for k, v in CONTENT_SHA.items()})

## 1 · Size and language

In [ ]:
size = pd.DataFrame(
    [{"dataset": ds, "split": s, "rows": len(d), **d.lang.value_counts().add_prefix("lang_").to_dict()} for (ds, s), d in DF.items()]
).fillna(0)
size["share_of_dataset"] = size.rows / size.groupby("dataset").rows.transform("sum")
size.to_csv(RUN_DIR / "size.csv", index=False)
display(size.round(3))

## 2 · Text: length, placeholders, audit flags
Lengths are whitespace word counts on `text_norm`, the shared model input. `[USER]` / `[URL]` are the placeholders written by 01. This section covers all three splits.

In [ ]:
USER_RE, URL_RE = re.compile(r"\[USER\]"), re.compile(r"\[URL\]")


def text_stats(d):
    words = d.text_norm.str.split().str.len()
    return {
        "words_mean": words.mean(), "words_median": words.median(), "words_p95": words.quantile(0.95), "words_max": words.max(),
        "chars_median": d.text_norm.str.len().median(),
        "share_with_user": d.text_norm.str.contains(USER_RE).mean(),
        "share_with_url": d.text_norm.str.contains(URL_RE).mean(),
        "empty_norm": int(d.empty_norm.sum()),
        "dup_rows": int(d.dup_group.notna().sum()),
        "leak_rows": int(d.leak_group.notna().sum()),
    }


text = pd.DataFrame([{"dataset": ds, "split": s, **text_stats(d)} for (ds, s), d in DF.items()])
text.to_csv(RUN_DIR / "text.csv", index=False)
display(text.round(3))

In [ ]:
# Share of posts per length bin (not counts), so splits of different size are comparable
clip = CONFIG["length_clip"]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=False)
for ax, ds in zip(axes, CONFIG["datasets"]):
    bins = np.arange(0, clip + 2, 2)
    for s in CONFIG["splits"]:
        w = DF[ds, s].text_norm.str.split().str.len().clip(upper=clip)
        ax.hist(w, bins=bins, weights=np.full(len(w), 1 / len(w)), histtype="step", linewidth=2, color=SPLIT_COLOR[s], label=f"{s} (n={len(w):,})")
    ax.set_title(ds.upper(), loc="left", fontsize=11)
    ax.set_xlabel(f"words in text_norm (≥{clip} in last bin)")
axes[0].set_ylabel("share of posts")
for ax in axes:
    ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(RUN_DIR / "figures" / "length_by_split.png", bbox_inches="tight")
plt.show()

## 3 · Binary label (train and dev)
`sexist` is the hard label. `sexist_soft` is the share of annotators voting sexist. CMSB is reported twice, on all posts and on annotated posts only (the main setting). The unannotated `other` posts are all non-sexist and come from a different source.

In [ ]:
LABELLED = [(ds, s) for ds in CONFIG["datasets"] for s in ["train", "dev"]]


def binary_row(ds, s, d, subset="all"):
    soft = d.sexist_soft.dropna()
    return {
        "dataset": ds, "split": s, "subset": subset, "rows": len(d),
        # Rate is over posts with a hard label (EXIST leaves out tweets with no majority)
        "hard_rows": int(d.sexist.notna().sum()), "sexist": int(d.sexist.sum()), "sexist_rate": d.sexist.mean(),
        "soft_n": len(soft),
        # Unanimity: every annotator agreed (soft 0 or 1)
        "unanimous_share": ((soft == 0) | (soft == 1)).mean() if len(soft) else None,
        "soft_mean": soft.mean() if len(soft) else None,
    }


rows = []
for ds, s in LABELLED:
    d = DF[ds, s]
    rows.append(binary_row(ds, s, d))
    if ds == "cmsb":
        rows.append(binary_row(ds, s, d[d.annotated], "annotated"))
binary = pd.DataFrame(rows)
binary.to_csv(RUN_DIR / "binary.csv", index=False)
display(binary.round(3))

In [ ]:
# Hard label against the annotator vote share: shows where the hard label isn't a plain majority
agree = []
for ds, s in LABELLED:
    d = DF[ds, s].dropna(subset=["sexist_soft"])
    vote_share = d.sexist_soft.round(3)
    agree.append(pd.crosstab([pd.Series(ds, index=d.index, name="dataset"), pd.Series(s, index=d.index, name="split"), vote_share.rename("sexist_soft")], d.sexist.rename("sexist")))
agree = pd.concat(agree).fillna(0).astype(int)
agree.to_csv(RUN_DIR / "binary_soft_vs_hard.csv")
display(agree)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, ds in zip(axes, CONFIG["datasets"]):
    vals = sorted(set(np.round(pd.concat([DF[ds, s].sexist_soft.dropna() for s in ["train", "dev"]]), 3)))
    x = np.arange(len(vals))
    for k, s in enumerate(["train", "dev"]):
        share = DF[ds, s].sexist_soft.dropna().round(3).value_counts(normalize=True).reindex(vals, fill_value=0)
        ax.bar(x + (k - 0.5) * 0.4, share.values, width=0.38, color=SPLIT_COLOR[s], label=s)
    ax.set_xticks(x, [f"{v:.2f}" for v in vals], fontsize=8)
    ax.set_title(f"{ds.upper()}: annotator share voting sexist", loc="left", fontsize=11)
    ax.legend(frameon=False, fontsize=8)
axes[0].set_ylabel("share of annotated posts")
fig.tight_layout()
fig.savefig(RUN_DIR / "figures" / "sexist_soft_by_split.png", bbox_inches="tight")
plt.show()

## 4 · CMSB structure
- `annotated` separates the five source subsets from the unannotated `other` posts.
- `of_id` marks adversarial modifications of an original post.
- Structure counts cover all splits. Label rates cover train and dev only.

In [ ]:
cmsb_rows = []
for s in CONFIG["splits"]:
    d = DF["cmsb", s]
    is_mod = d.of_id.notna() & (d.of_id != "-1")
    row = {"split": s, "rows": len(d), "modifications": int(is_mod.sum()), "originals": int((~is_mod).sum())}
    if s != "test":
        row.update({
            "annotated": int(d.annotated.sum()),
            "sexist_rate_originals": d.loc[~is_mod, "sexist"].mean(),
            "sexist_rate_modifications": d.loc[is_mod, "sexist"].mean(),
        })
    cmsb_rows.append(row)
cmsb_struct = pd.DataFrame(cmsb_rows)
cmsb_struct.to_csv(RUN_DIR / "cmsb_structure.csv", index=False)
display(cmsb_struct.round(3))

## 5 · EDOS Task B and Task C (train and dev)
Both are single-label over the sexist posts. "Category agreement" is the share of sexist posts whose hard Task B class got at least 2 of the 3 annotator votes.

In [ ]:
B, C = CONFIG["tasks"]["edos"]["task_b"]["classes"], CONFIG["tasks"]["edos"]["task_c"]["classes"]
edos_b, edos_c, edos_agree = [], [], []
for s in ["train", "dev"]:
    d = DF["edos", s]
    sx = d[d.sexist == 1]
    assert sx.category.notna().all() and d.loc[d.sexist == 0, "category"].isna().all(), "EDOS: category must be set exactly on sexist posts"
    edos_b.append(sx.category.value_counts().reindex(B, fill_value=0).rename(s))
    edos_c.append(sx.vector.value_counts().reindex(C, fill_value=0).rename(s))
    votes_for_hard = sx.apply(lambda r: r.category_votes[r.category], axis=1)
    edos_agree.append({"split": s, "sexist_posts": len(sx), "hard_class_votes_ge2": (votes_for_hard >= 2).mean(),
                       "hard_class_votes_eq3": (votes_for_hard == 3).mean()})


def counts_and_share(parts):
    t = pd.concat(parts, axis=1)
    return t.join((t / t.sum()).add_suffix("_share"))


edos_b, edos_c = counts_and_share(edos_b), counts_and_share(edos_c)
edos_agree = pd.DataFrame(edos_agree)
edos_b.to_csv(RUN_DIR / "edos_task_b.csv"); edos_c.to_csv(RUN_DIR / "edos_task_c.csv"); edos_agree.to_csv(RUN_DIR / "edos_agreement.csv", index=False)
display(edos_b.round(3), edos_c.round(3), edos_agree.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), gridspec_kw={"width_ratios": [4, 11]})
for ax, t, title in [(axes[0], edos_b, "Task B (category)"), (axes[1], edos_c, "Task C (vector)")]:
    x = np.arange(len(t))
    for k, s in enumerate(["train", "dev"]):
        ax.bar(x + (k - 0.5) * 0.4, t[f"{s}_share"], width=0.38, color=SPLIT_COLOR[s], label=s)
    ax.set_xticks(x, [i.replace("_", "\n") for i in t.index], fontsize=8)
    ax.set_title(f"EDOS {title}: share of sexist posts", loc="left", fontsize=11)
    ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(RUN_DIR / "figures" / "edos_b_c_by_split.png", bbox_inches="tight")
plt.show()

## 6 · EXIST Task 1.3 (train and dev)
Multi-label over 5 categories. A `labels` value of null means the tweet has no hard majority; these tweets are left out of the hard golds. `[]` means a not-sexist tweet. The label cardinality and co-occurrence below use sexist tweets with a hard label.

In [ ]:
X = CONFIG["tasks"]["exist"]["task_1_3"]["classes"]
ex_rows, ex_freq, ex_card, ex_soft = [], [], [], []
for s in ["train", "dev"]:
    d = DF["exist", s]
    hard = d[d.labels.notna()]
    n_lab = hard.labels.map(len)
    # Hard 1.3 should agree with hard 1.1 where both exist: [] exactly for not sexist
    both = hard[hard.sexist.notna()]
    ex_rows.append({
        "split": s, "rows": len(d), "no_hard_majority": int(d.labels.isna().sum()), "hard_not_sexist": int((n_lab == 0).sum()),
        "hard_sexist": int((n_lab > 0).sum()), "hard_1_1_vs_1_3_mismatch": int(((both.labels.map(len) > 0) != (both.sexist == 1)).sum()),
        "n_annot_ne_labels_n_annot": int((d.n_annot != d.labels_n_annot).sum()),
    })
    sx = hard[n_lab > 0]
    ex_freq.append(pd.Series({c: sx.labels.map(lambda v: c in v).sum() for c in X}, name=s))
    ex_card.append(sx.labels.map(len).value_counts().sort_index().rename(s))
    ex_soft.append(pd.DataFrame(d.labels_soft.tolist())[X].mean().rename(s))
exist_summary = pd.DataFrame(ex_rows)
exist_freq = counts_and_share(ex_freq)
exist_card = pd.concat(ex_card, axis=1).fillna(0).astype(int).rename_axis("labels_per_sexist_tweet")
exist_soft = pd.concat(ex_soft, axis=1).rename_axis("mean_soft_share")
for name, t in [("exist_summary", exist_summary), ("exist_label_freq", exist_freq), ("exist_cardinality", exist_card), ("exist_soft_mean", exist_soft)]:
    t.to_csv(RUN_DIR / f"{name}.csv", index=name != "exist_summary")
display(exist_summary, exist_freq.round(3), exist_card, exist_soft.round(3))

In [ ]:
# Co-occurrence: P(column label | row label) on hard-labelled sexist train tweets; the diagonal is 1
sx = DF["exist", "train"].dropna(subset=["labels"])
sx = sx[sx.labels.map(len) > 0]
M = np.array([[sx.labels.map(lambda v: a in v and b in v).sum() for b in X] for a in X])
cond = pd.DataFrame(M / M.diagonal()[:, None], index=X, columns=X)
cond.to_csv(RUN_DIR / "exist_cooccurrence_train.csv")

short = {"IDEOLOGICAL-INEQUALITY": "IDEO-INEQ", "STEREOTYPING-DOMINANCE": "STER-DOM", "OBJECTIFICATION": "OBJ",
         "SEXUAL-VIOLENCE": "SEX-VIOL", "MISOGYNY-NON-SEXUAL-VIOLENCE": "MIS-NSV"}
short = [short[c] for c in X]
fig, ax = plt.subplots(figsize=(5.6, 4.6))
from matplotlib.colors import LinearSegmentedColormap

# Reference sequential blue ramp, light to dark
blues = LinearSegmentedColormap.from_list("ref_blue", ["#f4f8fd", "#9ec5f4", "#3987e5", "#1c5cab", "#0d366b"])
im = ax.imshow(cond.values, cmap=blues, vmin=0, vmax=1)
for i in range(len(X)):
    for j in range(len(X)):
        v = cond.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color="white" if v > 0.55 else "#0b0b0b")
ax.set_xticks(range(len(X)), short, rotation=30, ha="right", fontsize=8)
ax.set_yticks(range(len(X)), short, fontsize=8)
ax.grid(False)
ax.set_title("EXIST train: P(column | row), hard labels", loc="left", fontsize=11)
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
fig.savefig(RUN_DIR / "figures" / "exist_cooccurrence_train.png", bbox_inches="tight")
plt.show()

## 7 · Length by label (train)
A quick check for length shortcuts: does the word-count distribution differ between sexist and not-sexist posts? For CMSB, annotated posts only.

In [ ]:
len_lab = []
for ds in CONFIG["datasets"]:
    d = DF[ds, "train"]
    d = d[d.annotated] if ds == "cmsb" else d
    d = d[d.sexist.notna()]
    w = d.text_norm.str.split().str.len()
    for v in [0, 1]:
        len_lab.append({"dataset": ds, "sexist": v, "rows": int((d.sexist == v).sum()),
                        "words_median": w[d.sexist == v].median(), "words_mean": w[d.sexist == v].mean()})
len_lab = pd.DataFrame(len_lab)
len_lab.to_csv(RUN_DIR / "length_by_label_train.csv", index=False)
display(len_lab.round(2))

## Report files
Two Markdown files are written to the run folder:
- `analysis.md`: every table above, plus the figures.
- `findings.md`: the findings. Every number in it is computed here from the tables, so the text always matches the run.

In [ ]:
def md_table(t, digits=3):
    """DataFrame to a GitHub Markdown table (no tabulate dependency)."""
    t = t.reset_index() if not isinstance(t.index, pd.RangeIndex) else t
    fmt = lambda v: f"{v:,.{digits}f}" if isinstance(v, float) else (f"{v:,}" if isinstance(v, (int, np.integer)) else str(v))
    rows = [[fmt(v) for v in r] for r in t.itertuples(index=False)]
    head = [str(c) for c in t.columns]
    return "\n".join(["| " + " | ".join(head) + " |", "|" + "---|" * len(head)] + ["| " + " | ".join(r) + " |" for r in rows])


SECTIONS = [
    ("1 · Size and language", [("Rows per split", size)], []),
    ("2 · Text (all splits)", [("Length, placeholders, audit flags", text)], ["length_by_split.png"]),
    ("3 · Binary label (train/dev)", [("Prevalence and agreement", binary), ("Annotator vote share vs hard label", agree)], ["sexist_soft_by_split.png"]),
    ("4 · CMSB structure", [("Originals and adversarial modifications", cmsb_struct)], []),
    ("5 · EDOS Task B / C (train/dev)", [("Task B", edos_b), ("Task C", edos_c), ("Category agreement", edos_agree)], ["edos_b_c_by_split.png"]),
    ("6 · EXIST Task 1.3 (train/dev)", [("Summary", exist_summary), ("Label frequency", exist_freq), ("Labels per sexist tweet", exist_card),
                                        ("Mean soft share", exist_soft), ("Co-occurrence P(column | row), train", cond)], ["exist_cooccurrence_train.png"]),
    ("7 · Length by label (train)", [("Words, not sexist vs sexist", len_lab)], []),
]
parts = [f"# Data analysis · {CONFIG['run_name']}\n",
         "Processed CMSB, EDOS, EXIST 2025 (`data/processed/`). Test is label-blind: only `TEST_COLS` are read. "
         f"Notebook `02_data_analysis.ipynb` (sha256 `{NOTEBOOK_SHA[:12]}`). Input content hashes are in `env.json`.\n"]
for title, tables, figs in SECTIONS:
    parts.append(f"## {title}\n")
    for name, t in tables:
        parts += [f"**{name}**\n", md_table(t), ""]
    parts += [f"![{f}](figures/{f})\n" for f in figs]
(RUN_DIR / "analysis.md").write_text("\n".join(parts))
print("wrote analysis.md")

In [ ]:
# Findings: wording is fixed, every number comes from this run
pct = lambda x: f"{100 * x:.1f}%"
B_ = binary.set_index(["dataset", "split", "subset"])
T_ = text.set_index(["dataset", "split"])
SZ = size.set_index(["dataset", "split"]).rows
rate = lambda ds, s, sub="all": B_.loc[(ds, s, sub), "sexist_rate"]
unan = lambda ds, s: B_.loc[(ds, s, "all"), "unanimous_share"]
span = lambda col, ds, f=pct: " to ".join(dict.fromkeys(f(T_.loc[(ds, s), col]) for s in sorted(CONFIG["splits"], key=lambda s: T_.loc[(ds, s), col])[::2]))
trip = lambda col, ds: " / ".join(f"{int(T_.loc[(ds, s), col]):,}" for s in CONFIG["splits"])


def edos_offmajority(s):
    # Hard label disagreeing with the 2-of-3 annotator majority
    d = DF["edos", s]
    return int(((d.sexist == 1) & (d.sexist_soft < 0.5)).sum() + ((d.sexist == 0) & (d.sexist_soft > 0.5)).sum()), len(d)


off = {s: edos_offmajority(s) for s in ["train", "dev"]}
EX = exist_summary.set_index("split")
CS = cmsb_struct.set_index("split")
EA = edos_agree.set_index("split")
LL = len_lab.set_index(["dataset", "sexist"]).words_median
small_c = edos_c["dev"][edos_c["dev"] < 10]
card = exist_card["train"] / exist_card["train"].sum()
pairs = cond.where(~np.eye(len(X), dtype=bool)).stack().sort_values(ascending=False).head(2)
b_share = edos_b["train_share"].sort_values(ascending=False)
x_share = exist_freq["train_share"].sort_values(ascending=False)
no_11 = {s: int(DF["exist", s].sexist.isna().sum()) for s in ["train", "dev"]}
med = lambda ds: "–".join(dict.fromkeys(f"{v:.0f}" for v in [T_.loc[ds, "words_median"].min(), T_.loc[ds, "words_median"].max()]))
# A vote share maps to one hard label when each soft value has a single non-zero column in the crosstab
one_to_one = {ds: bool(((agree.loc[ds] > 0).sum(axis=1) == 1).all()) for ds in CONFIG["datasets"]}

FINDINGS = f"""# Findings · {CONFIG['run_name']}
Observations from the processed data. Every number comes from the CSV tables in this folder (named in brackets). Test is label-blind: it appears only in the size, text and CMSB-structure counts.

## Size and text (`size.csv`, `text.csv`)
- All posts are English. Rows train / dev / test: CMSB {SZ['cmsb'].get('train'):,} / {SZ['cmsb'].get('dev'):,} / {SZ['cmsb'].get('test'):,}, EDOS {SZ['edos'].get('train'):,} / {SZ['edos'].get('dev'):,} / {SZ['edos'].get('test'):,}, EXIST (en) {SZ['exist'].get('train'):,} / {SZ['exist'].get('dev'):,} / {SZ['exist'].get('test'):,}.
- Median words in `text_norm` by split: CMSB {med('cmsb')}, EDOS {med('edos')}, EXIST {med('exist')}. The longest post has {int(text.words_max.max())} words. The length histograms of the three splits overlap closely (`figures/length_by_split.png`).
- `[USER]` appears in {span('share_with_user', 'cmsb')} of CMSB, {span('share_with_user', 'edos')} of EDOS and {span('share_with_user', 'exist')} of EXIST posts. `[URL]` appears in {span('share_with_url', 'cmsb')}, {span('share_with_url', 'edos')} and {span('share_with_url', 'exist')} of them.
- Audit flags, train / dev / test:
  - CMSB: placeholder-only {trip('empty_norm', 'cmsb')}, duplicate rows {trip('dup_rows', 'cmsb')}, cross-split leak rows {trip('leak_rows', 'cmsb')}.
  - EXIST: duplicates {trip('dup_rows', 'exist')}, leak rows {trip('leak_rows', 'exist')}.
  - EDOS: duplicates {trip('dup_rows', 'edos')}, leak rows {trip('leak_rows', 'edos')}.

## Binary label, train / dev (`binary.csv`, `binary_soft_vs_hard.csv`)
- Sexist rate:
  - CMSB: {pct(rate('cmsb', 'train'))} / {pct(rate('cmsb', 'dev'))} on all posts, and {pct(rate('cmsb', 'train', 'annotated'))} / {pct(rate('cmsb', 'dev', 'annotated'))} on annotated posts only.
  - EDOS: {pct(rate('edos', 'train'))} / {pct(rate('edos', 'dev'))}.
  - EXIST: {pct(rate('exist', 'train'))} / {pct(rate('exist', 'dev'))}, over tweets with a hard 1.1 label ({no_11['train']} / {no_11['dev']} tweets have no majority).
- Share of posts where all annotators agree: CMSB {pct(unan('cmsb', 'train'))} / {pct(unan('cmsb', 'dev'))}, EDOS {pct(unan('edos', 'train'))} / {pct(unan('edos', 'dev'))}, EXIST {pct(unan('exist', 'train'))} / {pct(unan('exist', 'dev'))}.
- EDOS hard labels differ from the 2-of-3 annotator majority on {off['train'][0]:,} / {off['train'][1]:,} train posts ({pct(off['train'][0] / off['train'][1])}) and {off['dev'][0]:,} / {off['dev'][1]:,} dev posts ({pct(off['dev'][0] / off['dev'][1])}). Does every vote share map to a single hard label? CMSB {'yes' if one_to_one['cmsb'] else 'no'}, EXIST {'yes' if one_to_one['exist'] else 'no'}, EDOS {'yes' if one_to_one['edos'] else 'no'}.

## CMSB structure (`cmsb_structure.csv`)
- Adversarial modifications, train / dev / test: {' / '.join(f"{int(CS.loc[s, 'modifications']):,}" for s in CONFIG['splits'])}.
- Train sexist rate: {pct(CS.loc['train', 'sexist_rate_originals'])} on originals vs {pct(CS.loc['train', 'sexist_rate_modifications'])} on modifications.

## EDOS Task B / C, train / dev (`edos_task_b.csv`, `edos_task_c.csv`, `edos_agreement.csv`)
- Task B shares of sexist posts (train): {', '.join(f"{c} {pct(v)}" for c, v in b_share.items())}.
- Task C classes with fewer than 10 dev examples: {', '.join(f"{c} ({int(v)})" for c, v in small_c.items()) or 'none'}.
- The hard Task B class has at least 2 of the 3 annotator votes on {pct(EA.loc['train', 'hard_class_votes_ge2'])} / {pct(EA.loc['dev', 'hard_class_votes_ge2'])} of sexist posts.

## EXIST Task 1.3, train / dev (`exist_*.csv`)
- Tweets with no hard 1.3 label: {int(EX.loc['train', 'no_hard_majority'])} / {int(EX.loc['dev', 'no_hard_majority'])}. Hard 1.1 vs 1.3 mismatches: {int(EX.loc['train', 'hard_1_1_vs_1_3_mismatch'])} / {int(EX.loc['dev', 'hard_1_1_vs_1_3_mismatch'])}.
- Labels per hard-labelled sexist train tweet: {', '.join(f"{k}: {pct(v)}" for k, v in card.items())}.
- Category shares (train): {', '.join(f"{c} {pct(v)}" for c, v in x_share.items())}.
- Strongest co-occurrences in train, P(column | row): {'; '.join(f"P({b} | {a}) = {v:.2f}" for (a, b), v in pairs.items())}.

## Length by label, train (`length_by_label_train.csv`)
- Median words, not sexist vs sexist: CMSB (annotated) {LL['cmsb', 0]:.0f} vs {LL['cmsb', 1]:.0f}, EDOS {LL['edos', 0]:.0f} vs {LL['edos', 1]:.0f}, EXIST {LL['exist', 0]:.0f} vs {LL['exist', 1]:.0f}.

## Interpretation (hedged)
- EXIST has the lowest annotator agreement of the three datasets, which suggests soft labels carry the most information there.
- The length gaps between classes are small, so length alone looks unlikely to be a strong shortcut. Only a trained length-only baseline would show this.
- Per-class dev thresholds for the small EDOS Task C classes will rest on very few examples.

## Open points
- Decide how to handle CMSB duplicates and cross-split leak rows (exclude, or report separately).
- Decide on the EDOS Task C thresholding policy, given the small dev classes.
- State which EXIST subset (hard-labelled only) each metric is computed on.
"""
(RUN_DIR / "findings.md").write_text(FINDINGS)
print(FINDINGS)

In [ ]:
# Run record, per the repo rules
TABLES = {
    "size": size, "text": text, "binary": binary, "cmsb_structure": cmsb_struct, "edos_agreement": edos_agree,
    "exist_summary": exist_summary, "length_by_label_train": len_lab,
}
METRICS = {k: t.to_dict("records") for k, t in TABLES.items()}
METRICS.update({"edos_task_b": edos_b.to_dict(), "edos_task_c": edos_c.to_dict(), "exist_label_freq": exist_freq.to_dict(),
                "exist_cardinality": exist_card.to_dict(), "exist_soft_mean": exist_soft.to_dict(), "exist_cooccurrence_train": cond.to_dict()})
(RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2) + "\n")
(RUN_DIR / "metrics.json").write_text(json.dumps(METRICS, indent=2, default=float) + "\n")
(RUN_DIR / "env.json").write_text(json.dumps({**ENV, "notebook_sha256": NOTEBOOK_SHA, "content_sha256": CONTENT_SHA}, indent=2) + "\n")
(RUN_DIR / "pip_freeze.txt").write_text(PKG_FREEZE)
(RUN_DIR / "README.md").write_text(f"""# {CONFIG['run_name']}
- Purpose: descriptive analysis of processed CMSB, EDOS, EXIST (train/dev/test). Test is label-blind (`TEST_COLS`).
- Notebook: `notebooks/02_data_analysis.ipynb` (sha256 `{NOTEBOOK_SHA[:12]}`); inputs' content hashes in `env.json`.
- Outcome: tables `*.csv`, figures in `figures/`, all tables in `analysis.md`, findings in `findings.md`, everything in `metrics.json`.
""")
print(sorted(p.name for p in RUN_DIR.rglob("*") if p.is_file()))

## Findings
The findings are written to `OUTPUT_ROOT/{run_name}/findings.md` by the report cell above. Its numbers are computed from this run's tables, and the cell prints it inline. The dated summary of each run is in `docs/experiment-log.md`.